# Kaputt Dataset: Exploration and Baseline Experiments

## 1. Introduction

This notebook introduces the **Kaputt** dataset and reproduces the key baseline experimental settings from the paper **"Kaputt: A Large-Scale Dataset for Visual Defect Detection" (ICCV 2025)**.

Kaputt is a large-scale benchmark for visual defect detection in retail logistics, containing:
- **238K+ images**
- **48K+ unique items**
- **29K+ defective instances**
- Around **40x larger than MVTec AD**

References:
- Paper: https://arxiv.org/abs/2510.05903
- Dataset website: https://www.kaputt-dataset.com/

In this notebook, we cover the four settings discussed in the paper:
1. **No training + No reference images** (zero-shot)
2. **No training + With reference images** (few-shot / reference-only memory)
3. **With training + No reference images**
4. **With training + With reference images**


## 2. Setup and Installation

Install dependencies for anomalib and parquet reading.

In [ ]:
%pip install anomalib[full]

In [ ]:
%matplotlib inline

import ast
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from rich.table import Table

from anomalib.data import Kaputt
from anomalib.data.datasets.image.kaputt import ImageMode, ImageType, KaputtDataset
from anomalib.engine import Engine
from anomalib.models import Patchcore, WinClip

DATA_ROOT = Path("../../../../datasets/kaputt")
SPLITS = ["train", "validation", "test"]

## 3. Dataset Exploration

This section demonstrates how to use both the Kaputt **datamodule/dataset API** and direct parquet loading for data profiling and visualization.

In [ ]:
# Datamodule API example
datamodule = Kaputt(root=DATA_ROOT, image_type=ImageType.CROP)
datamodule.setup()
print(f"Train samples: {len(datamodule.train_data)}")
print(f"Val samples: {len(datamodule.val_data)}")
print(f"Test samples: {len(datamodule.test_data)}")

# Dataset API example
kaputt_train_dataset = KaputtDataset(root=DATA_ROOT, split="train", image_type=ImageType.CROP)
print(f"KaputtDataset(train) samples: {len(kaputt_train_dataset)}")

In [ ]:
# Load parquet metadata for exploration
def load_parquet(split: str, kind: str = "query") -> pd.DataFrame:
    """Load parquet metadata for exploration."""
    path = DATA_ROOT / "datasets" / f"{kind}-{split}.parquet"
    if not path.exists():
        msg = f"Missing parquet file: {path}"
        raise FileNotFoundError(msg)
    return pd.read_parquet(path)


query_frames = {split: load_parquet(split, "query") for split in SPLITS}
reference_frames = {split: load_parquet(split, "reference") for split in SPLITS}

# Print as Table
table = Table(title="Kaputt Dataset Statistics")
table.add_column("Split")
table.add_column("Query Samples")
table.add_column("Reference Samples")

for split in SPLITS:
    table.add_row(split, str(len(query_frames[split])), str(len(reference_frames[split])))

table

In [ ]:
# Total number of samples per split (query and reference)
split_stats = pd.DataFrame({
    "split": SPLITS,
    "query_samples": [len(query_frames[s]) for s in SPLITS],
    "reference_samples": [len(reference_frames[s]) for s in SPLITS],
})
split_stats

In [ ]:
# Defect distribution: normal vs abnormal (query data)
all_query = pd.concat(query_frames.values(), ignore_index=True)
all_query["defect"] = all_query["defect"].astype(bool)
normal_abnormal = all_query["defect"].map({False: "normal", True: "abnormal"}).value_counts()

plt.figure(figsize=(6, 4))
normal_abnormal.plot(kind="bar", color=["#0071C5", "#5A42D8"])
plt.title("Defect distribution (normal vs abnormal)")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

normal_abnormal

In [ ]:
# Defect severity distribution: no defect, minor, major
severity = pd.Series("minor", index=all_query.index)
severity[~all_query["defect"]] = "no defect"
severity[all_query["major_defect"].astype(bool)] = "major"
severity_counts = severity.value_counts().reindex(["no defect", "minor", "major"], fill_value=0)

plt.figure(figsize=(7, 4))
severity_counts.plot(kind="bar", color=["#0071C5", "#00AEEF", "#5A42D8"])
plt.title("Defect severity distribution")
plt.xlabel("Severity")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

severity_counts

In [ ]:
# Defect type distribution
def parse_defect_types(value: list[str] | str | float | None) -> list:
    """Parse defect types from a variety of formats."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            return [value]
    return []


DEFECT_TYPES = ["penetration", "deformation", "actuation", "deconstruction", "spillage", "superficial", "missing_unit"]

defect_type_counts = (
    pd.Series([d for row in all_query["defect_types"].map(parse_defect_types) for d in row])
    .value_counts()
    .reindex(DEFECT_TYPES, fill_value=0)
)

plt.figure(figsize=(10, 4))
defect_type_counts.plot(kind="bar", color="#0071C5")
plt.title("Defect type distribution")
plt.xlabel("Defect type")
plt.ylabel("Count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

defect_type_counts

In [ ]:
# Material distribution
material_counts = all_query["item_material"].fillna("unknown").value_counts()

plt.figure(figsize=(10, 4))
material_counts.head(20).plot(kind="bar", color="#0071C5")
plt.title("Top-20 item material distribution")
plt.xlabel("Material")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

material_counts.head(20)

In [ ]:
# Number of reference images per split
reference_count_df = pd.DataFrame({
    "split": SPLITS,
    "reference_images": [len(reference_frames[s]) for s in SPLITS],
})

plt.figure(figsize=(6, 4))
plt.bar(reference_count_df["split"], reference_count_df["reference_images"], color="#0071C5")
plt.title("Reference images per split")
plt.xlabel("Split")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

reference_count_df

In [ ]:
# Utility to render image grids from dataset image paths
def show_images(image_paths: list[str], title: str, n: int = 6) -> None:
    """Show a grid of images from a list of file paths."""
    n = min(n, len(image_paths))
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, path in zip(axes, image_paths[:n], strict=False):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Grid of normal vs defective query images (using the datamodule)
train_samples = datamodule.train_data.samples

show_images(
    train_samples[train_samples["label"] == "normal"]["image_path"].tolist(),
    "Normal query samples",
    n=6,
)
show_images(
    train_samples[train_samples["label"] == "abnormal"]["image_path"].tolist(),
    "Defective query samples",
    n=6,
)

In [ ]:
# Query images and matching reference images by item_identifier
query_test = KaputtDataset(
    root=DATA_ROOT,
    split="test",
    image_type=ImageType.CROP,
    image_mode=ImageMode.QUERY_ONLY,
)
ref_test = KaputtDataset(
    root=DATA_ROOT,
    split="test",
    image_type=ImageType.CROP,
    image_mode=ImageMode.REFERENCE_ONLY,
)

pairs = query_test.samples[["item_identifier", "image_path"]].merge(
    ref_test.samples[["item_identifier", "image_path"]].drop_duplicates("item_identifier"),
    on="item_identifier",
    how="inner",
    suffixes=("_query", "_ref"),
)

n_pairs = min(6, len(pairs))
fig, axes = plt.subplots(n_pairs, 2, figsize=(8, 3 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for idx in range(n_pairs):
    row = pairs.iloc[idx]
    q_ax, r_ax = axes[idx]
    q_ax.imshow(Image.open(row["image_path_query"]).convert("RGB"))
    q_ax.set_title(f"Query | item={row['item_identifier']}")
    q_ax.axis("off")

    r_ax.imshow(Image.open(row["image_path_ref"]).convert("RGB"))
    r_ax.set_title("Reference")
    r_ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Visualize one sample per defect type (using the datamodule)
all_samples = pd.concat(
    [datamodule.train_data.samples, datamodule.val_data.samples, datamodule.test_data.samples],
    ignore_index=True,
)
defective = all_samples[all_samples["label"] == "abnormal"].copy()
defective["parsed_defect_types"] = defective["defect_types"].map(parse_defect_types)

picked_rows = []
for defect_name in DEFECT_TYPES:
    match = defective[defective["parsed_defect_types"].map(lambda x, dn=defect_name: dn in x)]
    if len(match) > 0:
        picked_rows.append((defect_name, match.iloc[0]["image_path"]))

n = len(picked_rows)
fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
if n == 1:
    axes = [axes]

for ax, (defect_name, path) in zip(axes, picked_rows, strict=False):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(defect_name)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Setting 1 — No Training, No Reference Images (WinCLIP Zero-Shot)

In this setup, **WinCLIP** uses CLIP image-text alignment and prompt-based anomaly scoring with **no model training** and **no reference images**.

In [ ]:
datamodule = Kaputt(root=DATA_ROOT, category="book_other", image_type=ImageType.CROP)
model = WinClip(class_name="object", k_shot=0)
engine = Engine()
results = engine.test(model=model, datamodule=datamodule)
print(results)

## 5. Setting 2 — No Training, With Reference Images

This setting uses reference images while avoiding supervised training on query labels.

### 5a. WinCLIP Few-Shot

WinCLIP in few-shot mode uses reference images as visual prototypes in addition to text prompts.

In [ ]:
datamodule = Kaputt(
    root=DATA_ROOT,
    category="book_other",
    image_type=ImageType.CROP,
    image_mode=ImageMode.REFERENCE_ONLY,
)
model = WinClip(class_name="object", k_shot=5)
engine = Engine()
results = engine.test(model=model, datamodule=datamodule)
print(results)

### 5b. PatchCore with Reference Images Only

PatchCore builds a memory bank from reference images only (no query training images).

In [ ]:
datamodule = Kaputt(
    root=DATA_ROOT,
    category="book_other",
    image_type=ImageType.CROP,
    image_mode=ImageMode.REFERENCE_ONLY,
)
model = Patchcore(backbone="resnet50", layers=["layer2", "layer3"])
engine = Engine()
engine.fit(model=model, datamodule=datamodule)
results = engine.test(model=model, datamodule=datamodule)
print(results)

## 6. Setting 3 — With Training, No Reference Images (PatchCore on Query Normals)

PatchCore is trained on normal **query** images only, without reference images.

In [ ]:
datamodule = Kaputt(root=DATA_ROOT, category="book_other", image_type=ImageType.CROP, image_mode=ImageMode.QUERY_ONLY)
model = Patchcore(backbone="resnet50", layers=["layer2", "layer3"])
engine = Engine()
engine.fit(model=model, datamodule=datamodule)
results = engine.test(model=model, datamodule=datamodule)
print(results)

## 7. Setting 4 — With Training and With Reference Images (PatchCore on Query + Reference)

PatchCore is trained on both normal query images and reference images, combining both normal-data sources.

In [ ]:
datamodule = Kaputt(
    root=DATA_ROOT,
    category="book_other",
    image_type=ImageType.CROP,
    image_mode=ImageMode.QUERY_AND_REFERENCE,
)
model = Patchcore(backbone="resnet50", layers=["layer2", "layer3"])
engine = Engine()
engine.fit(model=model, datamodule=datamodule)
results = engine.test(model=model, datamodule=datamodule)
print(results)